In [29]:
# 기본 import
import os
import xml.etree.ElementTree as ET
import pandas as pd
from glob import glob

# 결절 정보 추출

In [33]:
def extract_nodules(xml_path):

		# xml 읽기
    tree = ET.parse(xml_path)
    root = tree.getroot()

		# 결과 저장 리스트 생성
    nodules = []

    #  XML 전체 순회
    for session in root.iter():

				# readingSession만 선택
        tag = session.tag.split("}")[-1]

        if tag != "readingSession":
            continue

        # session 내부 탐색
        for nodule in session:
						
						# 실제 결절만 선택
            nodule_tag = nodule.tag.split("}")[-1]

            if nodule_tag != "unblindedReadNodule":
                continue

						# 결절 정보 저장 dict
            info = {}

            # nodule 내부 탐색
            for child in nodule:

								# nodule ID 추출
                child_tag = child.tag.split("}")[-1]

                if child_tag == "noduleID":
                    info["nodule_id"] = child.text

            nodules.append(info)

    return nodules

In [34]:
# 폴더 경로 지정
folder = "/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157"

#폴더 안 파일 목록 가져오기
xml_file = os.listdir(folder)[0]

# 전체 경로 만들기 (folder + xml_file)
xml_path = os.path.join(folder, xml_file)

print(xml_path)

/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/165.xml


In [35]:
# parsing 함수 실행
nodules = extract_nodules(xml_path)

print("nodule count:", len(nodules))
print(nodules[:5])

nodule count: 121
[{'nodule_id': '0'}, {'nodule_id': '2'}, {'nodule_id': '4'}, {'nodule_id': '5'}, {'nodule_id': '7'}]


# malignancy 추출

In [36]:
# xml 구조 확인
tree = ET.parse(xml_path)
root = tree.getroot()

for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "characteristics":

        for child in elem:

            print(child.tag, child.text)

        break

{http://www.nih.gov}subtlety 3
{http://www.nih.gov}internalStructure 1
{http://www.nih.gov}calcification 6
{http://www.nih.gov}sphericity 5
{http://www.nih.gov}margin 5
{http://www.nih.gov}lobulation 1
{http://www.nih.gov}spiculation 1
{http://www.nih.gov}texture 5
{http://www.nih.gov}malignancy 1


In [57]:
# 기존 함수에 characteristics 추가
def extract_nodules(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    nodules = []

    for session in root.iter():

        tag = session.tag.split("}")[-1]
        if tag != "readingSession":
            continue

        for nodule in session:

            nodule_tag = nodule.tag.split("}")[-1]
            if nodule_tag != "unblindedReadNodule":
                continue

            info = {}

            for child in nodule:

                child_tag = child.tag.split("}")[-1]

                # nodule ID
                if child_tag == "noduleID":
                    info["nodule_id"] = child.text

                # characteristics
                if child_tag == "characteristics":

                    for c in child:
                        ctag = c.tag.split("}")[-1]
                        info[ctag] = c.text

            # characteristics가 있는 경우만 추가
            if "malignancy" in info:
                nodules.append(info)

    return nodules

nodules = extract_nodules(xml_path)

print(len(nodules))
print(nodules[0])


21
{'nodule_id': 'CA006_12117', 'subtlety': '3', 'internalStructure': '1', 'calcification': '6', 'sphericity': '5', 'margin': '5', 'lobulation': '1', 'spiculation': '1', 'texture': '5', 'malignancy': '1'}


# roi_coords 추출

In [59]:
# XML 구조 확인
tree = ET.parse(xml_path)
root = tree.getroot()

for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "roi":

        print("FOUND ROI\n")

        for child in elem:
            print(child.tag)

        break

FOUND ROI

{http://www.nih.gov}imageZposition
{http://www.nih.gov}imageSOP_UID
{http://www.nih.gov}inclusion
{http://www.nih.gov}edgeMap


In [60]:
# edgeMap 구조 확인
for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "edgeMap":

        print("EDGE MAP")

        for child in elem:
            print(child.tag, child.text)

        break

EDGE MAP
{http://www.nih.gov}xCoord 306
{http://www.nih.gov}yCoord 255
